# Blinkit Data Analytics — SKU rationalisation
DuckDB SQL + Python | Yash Prajapati

## 1. Setup

### 1.1 Libraries

In [1]:
import warnings, math, textwrap
warnings.filterwarnings('ignore')
import duckdb, pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', lambda v: f'{v:,.2f}')
plt.rcParams.update({'figure.figsize':(10,4.5),'axes.grid':True,'grid.alpha':.25,'axes.spines.top':False,'axes.spines.right':False,'font.size':10})
print('duckdb', duckdb.__version__, '| pandas', pd.__version__, '| numpy', np.__version__)

duckdb 1.5.5 | pandas 3.0.2 | numpy 2.4.4


### 1.2 Load cleaned workbook

In [2]:
XLSX = 'data/Blinkit_analysis_new.xlsx'
book = pd.read_excel(XLSX, sheet_name=None)
geo = pd.read_csv('data/city_state_zone.csv')
for name, df in book.items():
    print(f'{name:28s} {df.shape[0]:>6,} rows  {df.shape[1]:>3} cols')

Data_Quality_Report              46 rows    3 cols
Orders_Customer_Info          5,000 rows   16 cols
Orders_Raw_Archive            5,000 rows   23 cols
Order_Line_Items              5,000 rows   14 cols
Delivery_Performance          5,000 rows    8 cols
Customer_Feedback             5,000 rows    8 cols
Customers                     2,500 rows   11 cols
Products                        268 rows   10 cols
Inventory_Analysis              268 rows   18 cols
Data_Dictionary                  44 rows    4 cols
Reference_Parameters             11 rows    5 cols
Relational_Diagram                0 rows    0 cols
Pivot_Payment_Method              7 rows    7 cols
Pivot_Customer_Segment            7 rows    5 cols
Pivot_Category_Sales             14 rows    4 cols
Pivot_Monthly_Trend              24 rows    6 cols
Pivot_Area_Delivery              23 rows    6 cols
Pivot_Inventory_Movement          6 rows    7 cols
Pivot_Category_Stock             14 rows    8 cols


### 1.3 Create DuckDB database and load tables

In [3]:
con = duckdb.connect('blinkit.duckdb')
load = {'orders_src':'Orders_Raw_Archive','items_src':'Order_Line_Items','delivery_src':'Delivery_Performance',
        'feedback_src':'Customer_Feedback','customers_src':'Customers','products_src':'Products','inventory_src':'Inventory_Analysis'}
for tbl, sheet in load.items():
    df = book[sheet].copy()
    df.columns = [c.strip() for c in df.columns]
    con.register('tmp_df', df)
    con.execute(f'CREATE OR REPLACE TABLE {tbl} AS SELECT * FROM tmp_df')
con.register('geo_df', geo)
con.execute('CREATE OR REPLACE TABLE geo AS SELECT * FROM geo_df')
con.execute("SELECT table_name, estimated_size FROM duckdb_tables() ORDER BY table_name").df()

,table_name,estimated_size
0,customers_src,2500
1,delivery_src,5000
2,feedback_src,5000
3,geo,316
4,inventory_src,268
5,items_src,5000
6,orders_src,5000
7,products_src,268


### 1.4 Typed analysis views

In [4]:
con.execute('''
CREATE OR REPLACE VIEW orders AS
SELECT order_id, customer_id,
       strptime(order_date, '%d-%m-%Y %H:%M')              AS order_ts,
       CAST(strptime(order_date, '%d-%m-%Y %H:%M') AS DATE) AS order_date,
       strptime(promised_delivery_time, '%d-%m-%Y %H:%M')   AS promised_ts,
       strptime(actual_delivery_time,  '%d-%m-%Y %H:%M')    AS actual_ts,
       delivery_status, order_total, payment_method, delivery_partner_id, store_id,
       CAST(delivery_time_minutes AS INTEGER)               AS delay_min,
       distance_km, reasons_if_delayed, customer_name,
       trim(area) AS area, pincode, customer_segment,
       CAST(registration_date AS DATE)                      AS registration_date,
       order_day_of_week, order_time_slot, order_value_segment,
       CASE WHEN is_weekend = 'Yes' THEN 1 ELSE 0 END       AS is_weekend
FROM orders_src ''')

con.execute('''
CREATE OR REPLACE VIEW items AS
SELECT order_id, product_id, quantity, unit_price, product_name, category, brand,
       price, mrp, margin_percentage, shelf_life_days, min_stock_level, max_stock_level, line_total,
       line_total * margin_percentage / 100.0 AS margin_value
FROM items_src ''')

con.execute('''
CREATE OR REPLACE VIEW feedback AS
SELECT feedback_id, order_id, customer_id, rating, feedback_category, sentiment,
       CAST(feedback_date AS DATE) AS feedback_date
FROM feedback_src ''')

con.execute('''
CREATE OR REPLACE VIEW f_sales AS
SELECT o.order_id, o.customer_id, o.order_ts, o.order_date,
       date_trunc('month', o.order_date)  AS order_month,
       extract(hour FROM o.order_ts)      AS order_hour,
       o.order_day_of_week, o.is_weekend, o.order_time_slot, o.order_value_segment,
       o.payment_method, o.customer_segment, o.registration_date, o.customer_name,
       o.area, g.state, g.zone, g.city_tier,
       i.product_id, i.product_name, i.category, i.brand,
       i.quantity, i.line_total AS revenue, i.margin_value, i.margin_percentage,
       i.price, i.mrp, i.shelf_life_days,
       o.delay_min, o.distance_km, o.delivery_status,
       CASE WHEN o.delivery_status = 'On Time' THEN 1 ELSE 0 END AS is_on_time_status,
       CASE WHEN o.delay_min > 0 THEN 1 ELSE 0 END               AS is_late_minutes,
       f.rating, f.sentiment, f.feedback_category
FROM orders o
JOIN items i    ON i.order_id = o.order_id
LEFT JOIN geo g ON g.area     = o.area
LEFT JOIN feedback f ON f.order_id = o.order_id ''')

con.execute('SELECT COUNT(*) AS fact_rows, COUNT(DISTINCT order_id) AS orders, MIN(order_date) AS first_day, MAX(order_date) AS last_day FROM f_sales').df()

,fact_rows,orders,first_day,last_day
0,5000,5000,2023-03-16,2024-11-04


### 1.5 Query helper

In [5]:
def q(sql, con=con):
    return con.execute(textwrap.dedent(sql)).df()

def pct(x, n):
    return round(100.0 * x / n, 2) if n else 0.0

q('SELECT COUNT(*) AS rows_in_fact_view FROM f_sales')

,rows_in_fact_view
0,5000


## 7. SKU rationalisation

### 7.1 Revenue contribution of the weakest product codes

In [6]:
q('''
WITH p AS (SELECT product_id, any_value(product_name) AS product_name, any_value(category) AS category,
                  SUM(revenue) AS revenue, COUNT(DISTINCT order_id) AS orders FROM f_sales GROUP BY product_id),
r AS (SELECT *, NTILE(4) OVER (ORDER BY revenue) AS quartile FROM p)
SELECT quartile, COUNT(*) AS product_codes, round(SUM(revenue), 0) AS revenue,
       round(100 * SUM(revenue) / SUM(SUM(revenue)) OVER (), 2) AS revenue_share_pct,
       round(AVG(orders), 1) AS avg_orders_per_code
FROM r GROUP BY quartile ORDER BY quartile ''')

,quartile,product_codes,revenue,revenue_share_pct,avg_orders_per_code
0,1,67,"286,298.00",5.76,17.90
1,2,67,"773,769.00",15.56,17.40
2,3,67,"1,442,179.00",29.00,18.20
3,4,67,"2,470,169.00",49.68,21.10


### 7.2 Merge and review candidates

In [7]:
cand = q('''
WITH p AS (SELECT product_id, any_value(product_name) AS product_name, any_value(category) AS category,
                  SUM(revenue) AS revenue, COUNT(DISTINCT order_id) AS orders FROM f_sales GROUP BY product_id),
n AS (SELECT product_name, COUNT(*) AS codes_for_name, SUM(revenue) AS name_revenue FROM p GROUP BY product_name)
SELECT p.product_id, p.product_name, p.category, p.orders, round(p.revenue, 0) AS revenue,
       n.codes_for_name, round(100 * p.revenue / n.name_revenue, 1) AS share_within_name,
       CASE WHEN n.codes_for_name > 1 AND 100 * p.revenue / n.name_revenue < 8 THEN 'merge into main code'
            WHEN p.orders <= 3 THEN 'review for removal'
            ELSE 'keep' END AS recommendation
FROM p JOIN n ON n.product_name = p.product_name ''')
cand.groupby('recommendation').agg(product_codes=('product_id','count'), orders=('orders','sum'),
    revenue=('revenue','sum')).assign(revenue_share_pct=lambda d: (100*d.revenue/cand.revenue.sum()).round(2)).reset_index()

,recommendation,product_codes,orders,revenue,revenue_share_pct
0,keep,194,3690,"4,532,097.00",91.14
1,merge into main code,74,1310,"440,320.00",8.86


### 7.3 Codes proposed for merging

In [8]:
cand[cand.recommendation == 'merge into main code'] \
    .sort_values('revenue').head(12)[['product_id','product_name','category','orders','revenue','codes_for_name','share_within_name']]

,product_id,product_name,category,orders,revenue,codes_for_name,share_within_name
52,118820,Iced Tea,Cold Drinks & Juices,14,370.00,3,0.50
191,654297,Potatoes,Fruits & Vegetables,17,477.00,7,0.50
26,962054,Orange Juice,Cold Drinks & Juices,16,506.00,7,0.50
124,652118,Frozen Biryani,Instant & Frozen Food,16,749.00,7,0.70
260,767398,Dish Soap,Household Care,14,753.00,10,0.40
37,300159,Cola,Cold Drinks & Juices,12,931.00,7,0.70
161,709916,Detergent,Household Care,15,"1,148.00",7,1.90
146,133542,Detergent,Household Care,9,"1,164.00",7,1.90
261,473647,Toothpaste,Personal Care,15,"1,291.00",5,1.30
248,968887,Dish Soap,Household Care,19,"1,587.00",10,0.90
